In [ ]:
import operator
import re
from dataclasses import dataclass
from typing import Annotated, Any, Literal

from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from kagraph import END, START, InMemorySaver, Send, StateGraph, add_messages, prompt_llm
from kagraph.llms import load_llm
from kagraph.messages import AIMessage, AnyMessage, HumanMessage
from kagraph.runtime import Runtime
from kagraph.tracing import trace

In [ ]:
OP_FUNCS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
}
OPERATORS = set(OP_FUNCS)
OperatorType = Literal["+", "-", "*", "/"]

In [ ]:
@dataclass(frozen=True)
class Equation:
    """Reverse-polish notation equation for the Game of 24."""

    tokens: list[float | str]

    def compute(self) -> float:
        stack: list[float] = []
        for token in self.tokens:
            if isinstance(token, (int, float)):
                stack.append(float(token))
                continue
            if token not in OP_FUNCS:
                raise ValueError(f"Unsupported operator {token!r}")
            if len(stack) < 2:
                raise ValueError(f"Operator {token!r} does not have two operands.")
            b = stack.pop()
            a = stack.pop()
            stack.append(OP_FUNCS[token](a, b))
        if len(stack) != 1:
            raise ValueError(f"Expression leaves {len(stack)} values on stack.")
        return stack[0]


@dataclass(frozen=True)
class Candidate:
    candidate: Equation
    score: float | None = None
    feedback: str | None = None

    def __str__(self) -> str:
        try:
            computed = self.candidate.compute()
        except Exception as error:
            computed = f"Invalid equation: {self.candidate.tokens}; Error: {error!r}"
        return f"Equation({self.candidate.tokens}) = {computed} (Reward: {self.score})"

In [ ]:
class EquationGuess(BaseModel):
    """One reverse-polish notation equation candidate for the Game of 24."""

    tokens: list[float | OperatorType] = Field(
        description=(
            "Reverse-polish notation tokens. Use numbers from the puzzle and "
            "operator strings from '+', '-', '*', '/'. Example: [12, 5, '/', 7, '*']."
        )
    )


class GuessEquations(BaseModel):
    """A batch of candidate equations submitted by the expander."""

    reasoning: str = Field(description="Brief reasoning behind the candidate equations.")
    equations: list[EquationGuess] = Field(description="Candidate RPN equations.")


def update_candidates(
    existing: list[Candidate] | None = None,
    updates: list[Candidate] | Literal["clear"] | None = None,
) -> list[Candidate]:
    if existing is None:
        existing = []
    if updates is None:
        return existing
    if updates == "clear":
        return []
    return existing + updates

In [ ]:
class ToTInput(TypedDict):
    problem: str


class ToTState(TypedDict, total=False):
    problem: str
    messages: Annotated[list[AnyMessage], add_messages]
    candidates: Annotated[list[Candidate], update_candidates]
    scored_candidates: Annotated[list[Candidate], update_candidates]
    depth: Annotated[int, operator.add]
    seed: Candidate
    answer: str


class Context(TypedDict, total=False):
    max_depth: int
    threshold: float
    k: int
    beam_size: int


EXPAND_PROMPT = """You are playing the Game of 24.

Given four numbers, propose candidate equations in reverse-polish notation (RPN).
The final goal is an RPN expression that uses each number exactly once and evaluates to 24.

Puzzle numbers: {problem}

Important:
- Return exactly {k} candidates.
- Allowed operators: "+", "-", "*", "/".
- Partial candidates are allowed. They may use only a subset of the four numbers.
- If a seed candidate is provided, improve or extend from that seed.
- Submit candidates through the structured output schema.

{seed_text}
"""

In [ ]:
def _context(runtime: Runtime) -> dict[str, Any]:
    ctx = runtime.context
    return {
        "max_depth": int(ctx.get("max_depth", 10)),
        "threshold": float(ctx.get("threshold", 0.9)),
        "k": int(ctx.get("k", 5)),
        "beam_size": int(ctx.get("beam_size", 3)),
    }


def _candidates_from_submission(submission: GuessEquations | dict[str, Any]) -> list[Candidate]:
    if isinstance(submission, dict):
        submission = GuessEquations.model_validate(submission)
    return [
        Candidate(Equation(_coerce_tokens(list(equation.tokens))))
        for equation in submission.equations
    ]


def _coerce_tokens(raw: list[Any]) -> list[float | str]:
    tokens: list[float | str] = []
    for token in raw:
        if isinstance(token, (int, float)):
            tokens.append(float(token))
        elif isinstance(token, str) and token in OPERATORS:
            tokens.append(token)
        else:
            raise ValueError(f"Unsupported RPN token {token!r} in {raw!r}")
    return tokens


def _numbers_from_problem(problem: str) -> list[float]:
    numbers = [float(match) for match in re.findall(r"-?\d+(?:\.\d+)?", problem)]
    if len(numbers) != 4:
        raise ValueError(f"Expected exactly four numbers, got {numbers!r}")
    return numbers


def _numbers_used(candidate: Candidate) -> list[float]:
    return [float(token) for token in candidate.candidate.tokens if isinstance(token, (int, float))]


def compute_score(problem: str, candidate: Candidate) -> Candidate:
    expected = sorted(_numbers_from_problem(problem))
    used = sorted(_numbers_used(candidate))
    if used != expected:
        return Candidate(
            candidate.candidate,
            0,
            "The equation must use all 4 numbers exactly once.",
        )
    try:
        result = candidate.candidate.compute()
        score = 1 / (1 + abs(24 - result))
        feedback = f"Result: {result}"
    except Exception as error:
        score = 0
        feedback = f"Invalid equation. Error: {error!r}"
    return Candidate(candidate.candidate, score, feedback)

In [ ]:
def build_graph(llm: Any):
    graph = StateGraph(ToTState, input_schema=ToTInput, context_schema=Context)

    def prepare_problem(state: ToTInput):
        return {"messages": [HumanMessage(f"Solve the Game of 24 puzzle: {state['problem']}")]}

    def expand(state: ToTState, *, runtime: Runtime):
        ctx = _context(runtime)
        seed = state.get("seed")
        seed_text = f"Seed candidate to improve:\n{seed}\n" if seed is not None else ""
        prompt = EXPAND_PROMPT.format(
            problem=state["problem"],
            k=ctx["k"],
            seed_text=seed_text,
        )
        submission = prompt_llm(
            llm,
            prompt,
            schema=GuessEquations,
        )
        return {"candidates": _candidates_from_submission(submission)}

    def score(state: ToTState):
        scored = [
            compute_score(state["problem"], candidate)
            for candidate in state.get("candidates", [])
        ]
        return {"scored_candidates": scored, "candidates": "clear"}

    def prune(state: ToTState, *, runtime: Runtime):
        ctx = _context(runtime)
        organized = sorted(
            state.get("scored_candidates", []),
            key=lambda candidate: candidate.score or 0,
            reverse=True,
        )
        return {
            "candidates": organized[: ctx["beam_size"]],
            "scored_candidates": "clear",
            "depth": 1,
        }

    def should_terminate(state: ToTState, runtime: Runtime):
        ctx = _context(runtime)
        candidates = state.get("candidates", [])
        solved = bool(candidates) and (candidates[0].score or 0) >= ctx["threshold"]
        if solved or not candidates or state.get("depth", 0) >= ctx["max_depth"]:
            return "final_answer"
        return [Send("expand", {**state, "seed": candidate}) for candidate in candidates]

    def final_answer(state: ToTState, *, runtime: Runtime):
        ctx = _context(runtime)
        candidates = state.get("candidates", [])
        if not candidates:
            answer = "I can't solve this puzzle because no valid candidates were generated."
            return {"answer": answer, "messages": [AIMessage(answer)]}

        best = candidates[0]
        score = best.score or 0
        if score >= ctx["threshold"]:
            answer = f"Solved: {best}"
            return {"answer": answer, "messages": [AIMessage(answer)]}

        answer = (
            "I can't solve this puzzle within the configured search limits. "
            f"Best candidate: {best}"
        )
        return {"answer": answer, "messages": [AIMessage(answer)]}

    graph.add_node("prepare_problem", prepare_problem, input_schema=ToTInput)
    graph.add_node("expand", expand)
    graph.add_node("score", score)
    graph.add_node("prune", prune)
    graph.add_node("final_answer", final_answer)

    graph.add_edge(START, "prepare_problem")
    graph.add_edge("prepare_problem", "expand")
    graph.add_edge("expand", "score")
    graph.add_edge("score", "prune")
    graph.add_conditional_edges(
        "prune",
        should_terminate,
        {
            "expand": "expand",
            "final_answer": "final_answer",
        },
    )
    graph.add_edge("final_answer", END)

    return graph.compile(name="tot_game_of_24", checkpointer=InMemorySaver())

In [ ]:
llm = load_llm("qwen/qwen3-next-80b-a3b-instruct", support_structured_outputs=True)
app = build_graph(llm)
config = {"configurable": {"thread_id": "tot-game-of-24-12-1-5-7"}}

In [ ]:
app.get_graph().draw_png()

In [ ]:
with trace("TreeOfThought"):
    result = app.invoke(
        {"problem": "12 1 5 7"},
        config=config,
        context={"max_depth": 10, "threshold": 0.9, "k": 5, "beam_size": 3},
        chat_name="KaGraph Tree of Thought Game of 24 example",
        recursion_limit=80,
)

: 